# 外汇汇率预测 Transformer 模型

本笔记演示了如何使用 PyTorch 构建 Transformer 模型，对 Excel 中的外汇汇率特征进行建模、训练与预测，同时包含自动超参数搜索、指标评估与可视化分析。

## 使用说明
1. 在下方的 **Configuration** 区块中，根据自己的 Excel 文件路径及字段，指定因变量列 (`CONFIG['target_columns']`)；未列出的（除日期列外）其余列默认作为自变量。
2. 确保 Excel 文件中包含 `Date` 日期列，或在配置中指定正确的日期列名称。
3. 通过 `CONFIG['hyperparameter_search']` 调整搜索策略、评估次数与候选空间；默认使用贝叶斯优化，如需运行请确保已安装 `scikit-optimize`（`pip install scikit-optimize`）。
4. 依次运行各个代码单元格以完成数据加载、特征工程、超参数搜索、模型训练、评估与可视化。


In [ ]:
# Configuration
from pathlib import Path

CONFIG = {
    "excel_path": Path("data/forex_features.xlsx"),  # Excel 文件路径
    "date_column": "Date",  # 日期列名称
    "target_columns": [
        # "Exchange_Rate",  # 在此列出需要预测的因变量列名
    ],
    "test_ratio": 0.2,  # 测试集比例
    "val_ratio": 0.1,  # 验证集比例（用于滚动验证）
    "lookback": 30,  # 序列窗口长度
    "horizon": 1,  # 预测步长（1 表示预测下一期）
    "epochs": 100,
    "seed": 42,
    "device": "cuda",  # 如果有 GPU，可设置为 "cuda"，否则保持 "cpu"
    "default_hyperparameters": {
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "dim_feedforward": 128,
        "dropout": 0.1,
        "batch_size": 32,
        "learning_rate": 1e-3,
    },
    "hyperparameter_search": {
        "strategy": "bayesian",  # 可选 "bayesian" 或 "grid"
        "n_trials": 20,  # 贝叶斯优化评估的最大次数
        "n_initial_points": 5,  # (可选) 初始随机探索次数
        "acq_func": "EI",  # (可选) 采集函数，可设为 "PI"、"LCB" 等
        "param_space": {
            "d_model": [64, 96, 128],
            "nhead": [4, 8],
            "num_layers": [2, 3],
            "dim_feedforward": [128, 256],
            "dropout": [0.1, 0.2],
            "batch_size": [32, 64],
            "learning_rate": [1e-3, 5e-4, 2e-4],
        },
    },
    "gradient_clip": 1.0,
    "lr_scheduler": {
        "enabled": True,
        "factor": 0.5,
        "patience": 5,
        "threshold": 1e-4,
        "min_lr": 1e-6,
        "cooldown": 0,
        "verbose": False,
    },
    "early_stopping": {
        "enabled": True,
        "patience": 10,
        "min_delta": 1e-4,
    },
    "rolling_validation": {
        "enabled": True,
        "window_size": None,  # 可设为整数或比例（0-1），控制每个验证窗口的长度
        "step_size": None,  # 可设为整数或比例（0-1），控制验证窗口的步进
        "max_windows": 5,  # 最多生成的滚动验证窗口数量
        "min_train_size": None,  # (可选) 最小训练样本行数，默认自动推断
    },
    "directional_threshold": 0.0,  # 方向命中率计算时忽略小于该阈值的变动
}


In [ ]:
# Imports
import copy
import math
import inspect
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import ParameterGrid
import matplotlib.pyplot as plt

try:
    from skopt import gp_minimize
    from skopt.space import Real, Integer, Categorical
    from skopt.utils import use_named_args
except ImportError:  # pragma: no cover
    gp_minimize = None
    Real = Integer = Categorical = None

    def use_named_args(*_args, **_kwargs):
        raise ImportError("需要安装 scikit-optimize 才能使用贝叶斯优化。请运行 `pip install scikit-optimize`。")

plt.style.use("seaborn-v0_8")

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
device = torch.device(CONFIG["device"] if torch.cuda.is_available() and CONFIG["device"] == "cuda" else "cpu")
print(f"Using device: {device}")


In [ ]:
# Dataset Utilities
class SequenceDataset(Dataset):
    def __init__(self, features, targets, name=None):
        if len(features) != len(targets):
            raise ValueError(f"{name or 'dataset'} 的特征与标签数量不一致。")
        if len(features) == 0:
            raise ValueError(f"{name or 'dataset'} 在当前窗口参数下没有可用样本。")
        self.features = torch.as_tensor(features, dtype=torch.float32)
        self.targets = torch.as_tensor(targets, dtype=torch.float32)
        self.name = name or 'dataset'

    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]


def build_sequence_dataset(
    df,
    *,
    feature_scaler,
    target_scaler,
    lookback,
    horizon,
    feature_cols,
    target_cols,
    dataset_name='dataset',
    context_df=None,
):
    if df is None or len(df) == 0:
        raise ValueError(f"{dataset_name} 数据为空，无法构建序列。")
    if lookback <= 0 or horizon <= 0:
        raise ValueError('lookback 与 horizon 必须为正整数。')

    if context_df is not None:
        if len(context_df) < lookback:
            raise ValueError(
                f"{dataset_name} 的 context_df 行数 ({len(context_df)}) 小于 lookback={lookback}，无法提供足够的历史窗口。"
            )
        context_tail = context_df.tail(lookback)
        combined_df = pd.concat([context_tail, df], axis=0)
        context_len = len(context_tail)
    else:
        combined_df = df
        context_len = 0

    feature_values = feature_scaler.transform(combined_df[feature_cols])
    target_values = target_scaler.transform(combined_df[target_cols])

    total_rows = len(combined_df)
    max_start = total_rows - lookback - horizon + 1
    if max_start <= 0:
        raise ValueError(
            f"{dataset_name} 无法在 lookback={lookback}, horizon={horizon} 设置下生成序列，请检查数据量。"
        )

    sequences = []
    targets = []
    for start in range(0, max_start):
        target_idx = start + lookback + horizon - 1
        if context_len and target_idx < context_len:
            continue
        seq = feature_values[start : start + lookback]
        tgt = target_values[target_idx]
        sequences.append(np.asarray(seq, dtype=np.float32))
        targets.append(np.asarray(tgt, dtype=np.float32))

    if not sequences:
        raise ValueError(
            f"{dataset_name} 在当前窗口参数下没有生成有效的序列，请调整切分或窗口设置。"
        )

    features_np = np.stack(sequences)
    targets_np = np.stack(targets)
    return SequenceDataset(features_np, targets_np, name=dataset_name)



In [ ]:
# Data Loading
excel_path = Path(CONFIG["excel_path"])
if not excel_path.exists():
    raise FileNotFoundError(f"未找到 Excel 文件: {excel_path.resolve()}")

date_col = CONFIG["date_column"]
raw_df = pd.read_excel(excel_path, parse_dates=[date_col])
if raw_df[date_col].isna().any():
    raise ValueError("日期列包含缺失值，请清理数据后再运行。")

candidate_target_cols = CONFIG.get("target_columns")
if not candidate_target_cols:
    raise ValueError("请在 CONFIG['target_columns'] 中至少指定一个因变量列名。")

if isinstance(candidate_target_cols, str):
    candidate_target_cols = [candidate_target_cols]
else:
    candidate_target_cols = list(candidate_target_cols)

missing_targets = [col for col in candidate_target_cols if col not in raw_df.columns]
if missing_targets:
    raise ValueError(f"以下因变量列在数据中不存在: {missing_targets}")

df = raw_df.drop_duplicates(subset=[date_col]).set_index(date_col).sort_index()
target_cols = list(dict.fromkeys(candidate_target_cols))
feature_cols = [col for col in df.columns if col not in target_cols]

if not feature_cols:
    raise ValueError("除因变量外没有其他列可作为自变量，请检查数据。")

print(f"Target columns: {target_cols}")
print(f"Feature columns: {feature_cols}")

display(df.head())


In [ ]:
# Descriptive Statistics
selected_cols = list(dict.fromkeys(feature_cols + target_cols))
display(df[selected_cols].describe().T)
display(pd.DataFrame({"missing_values": df[selected_cols].isna().sum()}))


In [ ]:
# Train/Validation/Test Split and Scaling
test_ratio = CONFIG["test_ratio"]
val_ratio = CONFIG["val_ratio"]
lookback = CONFIG["lookback"]
horizon = CONFIG["horizon"]

if not 0 < test_ratio < 1:
    raise ValueError("test_ratio 必须在 0 与 1 之间。")

if not 0 < val_ratio < 1:
    raise ValueError("val_ratio 必须在 0 与 1 之间。")

if test_ratio + val_ratio >= 1:
    raise ValueError("test_ratio 与 val_ratio 之和必须小于 1。")

if lookback <= 0:
    raise ValueError("lookback 必须为正整数。")

if horizon <= 0:
    raise ValueError("horizon 必须为正整数。")

total_len = len(df)
test_size = max(int(total_len * test_ratio), horizon)
val_size = max(int(total_len * val_ratio), horizon)
train_size = total_len - val_size - test_size

if train_size <= lookback:
    raise ValueError("训练集样本过少，无法满足 lookback 要求，请调整数据或参数。")

train_df = df.iloc[:train_size].copy()
val_df = df.iloc[train_size:train_size + val_size].copy()
test_df = df.iloc[train_size + val_size :].copy()

if len(val_df) < horizon:
    raise ValueError("验证集样本过少，无法生成序列，请增加 val_ratio。")

if len(test_df) < horizon:
    raise ValueError("测试集样本过少，无法生成序列，请调整 test_ratio 或 horizon。")

combined_train_val = pd.concat([train_df, val_df])

rolling_cfg = CONFIG.get("rolling_validation", {})
use_rolling = bool(rolling_cfg.get("enabled", True))


def _resolve_window_size(value, total_length, default_size):
    if value is None:
        return default_size
    if isinstance(value, float) and 0 < value < 1:
        resolved = int(math.ceil(total_length * value))
    else:
        resolved = int(value)
    if resolved <= 0:
        raise ValueError("rolling_validation 的窗口或步长需为正数。")
    return resolved

if len(val_df) >= horizon * 3:
    default_window = max(horizon, len(val_df) // 3)
else:
    default_window = max(horizon, len(val_df))

window_size = _resolve_window_size(rolling_cfg.get("window_size"), len(val_df), default_window)
step_default = window_size if len(val_df) <= window_size else max(1, window_size // 2)
step_size = _resolve_window_size(rolling_cfg.get("step_size"), len(val_df), step_default)
max_windows = int(rolling_cfg.get("max_windows", 5))
if max_windows <= 0:
    raise ValueError("rolling_validation['max_windows'] 必须为正整数。")

min_train_size_cfg = rolling_cfg.get("min_train_size")
if min_train_size_cfg is None:
    min_train_size = max(lookback + horizon, lookback + 1)
else:
    min_train_size = int(min_train_size_cfg)
    if min_train_size <= lookback:
        raise ValueError("rolling_validation['min_train_size'] 必须大于 lookback。")

start_positions = [0]
if len(val_df) > window_size:
    start_positions = list(range(0, len(val_df) - window_size + 1, step_size))
    last_start = len(val_df) - window_size
    if start_positions[-1] != last_start:
        start_positions.append(last_start)

rolling_folds = []

if use_rolling and len(val_df) >= horizon:
    for start in start_positions:
        train_end = train_df.shape[0] + start
        train_slice = combined_train_val.iloc[:train_end]
        if len(train_slice) <= lookback or len(train_slice) < min_train_size:
            continue

        val_slice = val_df.iloc[start : start + window_size]

        feature_scaler = StandardScaler().fit(train_slice[feature_cols])
        target_scaler = StandardScaler().fit(train_slice[target_cols])

        train_dataset_fold = build_sequence_dataset(
            train_slice,
            feature_scaler=feature_scaler,
            target_scaler=target_scaler,
            lookback=lookback,
            horizon=horizon,
            feature_cols=feature_cols,
            target_cols=target_cols,
            dataset_name=f"train_fold_{len(rolling_folds) + 1}",
        )

        val_context = train_slice.tail(lookback)
        val_dataset_fold = build_sequence_dataset(
            val_slice,
            feature_scaler=feature_scaler,
            target_scaler=target_scaler,
            lookback=lookback,
            horizon=horizon,
            feature_cols=feature_cols,
            target_cols=target_cols,
            context_df=val_context,
            dataset_name=f"validation_fold_{len(rolling_folds) + 1}",
        )

        rolling_folds.append(
            {
                "train_dataset": train_dataset_fold,
                "val_dataset": val_dataset_fold,
                "train_rows": len(train_slice),
                "val_rows": len(val_slice),
            }
        )

        if len(rolling_folds) >= max_windows:
            break

if not rolling_folds:
    fallback_scaler_features = StandardScaler().fit(train_df[feature_cols])
    fallback_scaler_targets = StandardScaler().fit(train_df[target_cols])

    train_dataset = build_sequence_dataset(
        train_df,
        feature_scaler=fallback_scaler_features,
        target_scaler=fallback_scaler_targets,
        lookback=lookback,
        horizon=horizon,
        feature_cols=feature_cols,
        target_cols=target_cols,
        dataset_name="train",
    )

    val_context = train_df.tail(lookback)
    val_dataset = build_sequence_dataset(
        val_df,
        feature_scaler=fallback_scaler_features,
        target_scaler=fallback_scaler_targets,
        lookback=lookback,
        horizon=horizon,
        feature_cols=feature_cols,
        target_cols=target_cols,
        context_df=val_context,
        dataset_name="validation",
    )

    rolling_folds.append(
        {
            "train_dataset": train_dataset,
            "val_dataset": val_dataset,
            "train_rows": len(train_df),
            "val_rows": len(val_df),
        }
    )
else:
    train_dataset = rolling_folds[0]["train_dataset"]
    val_dataset = rolling_folds[0]["val_dataset"]

print(f"Train rows: {len(train_df)} | Validation rows: {len(val_df)} | Test rows: {len(test_df)}")
print(f"Rolling validation folds: {len(rolling_folds)} (window size ≈ {window_size} 行, step ≈ {step_size} 行)")
for idx, fold in enumerate(rolling_folds, start=1):
    print(
        f"  Fold {idx}: train rows = {fold['train_rows']} (sequences {len(fold['train_dataset'])}), "
        f"validation rows = {fold['val_rows']} (sequences {len(fold['val_dataset'])})"
    )


In [ ]:
# Model Definition & Training Utilities
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


class TimeSeriesTransformer(nn.Module):
    def __init__(
        self,
        *,
        input_size,
        output_size,
        d_model,
        nhead,
        num_layers,
        dim_feedforward,
        dropout,
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.positional_encoding = PositionalEncoding(d_model=d_model, dropout=dropout)
        self.regressor = nn.Linear(d_model, output_size)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x)
        output = self.regressor(x[:, -1, :])
        return output


def instantiate_model(params):
    return TimeSeriesTransformer(
        input_size=len(feature_cols),
        output_size=len(target_cols),
        d_model=params["d_model"],
        nhead=params["nhead"],
        num_layers=params["num_layers"],
        dim_feedforward=params["dim_feedforward"],
        dropout=params["dropout"],
    ).to(device)


def build_scheduler(optimizer):
    scheduler_cfg = CONFIG.get("lr_scheduler", {})
    if not isinstance(scheduler_cfg, dict) or not scheduler_cfg.get("enabled", True):
        return None

    factor = float(scheduler_cfg.get("factor", 0.5))
    patience = int(scheduler_cfg.get("patience", 5))
    threshold = float(scheduler_cfg.get("threshold", 1e-4))
    cooldown = int(scheduler_cfg.get("cooldown", 0))
    min_lr = float(scheduler_cfg.get("min_lr", 1e-6))
    verbose = bool(scheduler_cfg.get("verbose", False))

    scheduler_kwargs = dict(
        mode="min",
        factor=factor,
        patience=patience,
        threshold=threshold,
        cooldown=cooldown,
        min_lr=min_lr,
    )

    signature = inspect.signature(ReduceLROnPlateau.__init__)
    if "verbose" in signature.parameters:
        scheduler_kwargs["verbose"] = verbose
    elif verbose:
        print("警告: 当前 PyTorch 版本的 ReduceLROnPlateau 不支持 verbose 参数，将忽略该配置。")

    return ReduceLROnPlateau(optimizer, **scheduler_kwargs)


def run_training(
    model,
    optimizer,
    train_loader,
    val_loader,
    epochs,
    log_prefix="",
    gradient_clip=None,
    scheduler=None,
):
    history = {"train_loss": [], "val_loss": [], "lr": []}
    gradient_clip = gradient_clip if gradient_clip is not None else CONFIG.get("gradient_clip", 1.0)
    display_step = max(1, epochs // 10)

    early_cfg = CONFIG.get("early_stopping", {})
    use_early_stopping = bool(early_cfg.get("enabled", True) and val_loader is not None)
    patience = int(early_cfg.get("patience", 10))
    min_delta = float(early_cfg.get("min_delta", 0.0))
    wait = 0

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            preds = model(batch_x)
            loss = nn.functional.mse_loss(preds, batch_y)
            loss.backward()
            if gradient_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
            optimizer.step()

            train_loss += loss.item() * batch_x.size(0)

        train_loss /= len(train_loader.dataset)
        history["train_loss"].append(train_loss)

        stop_now = False
        if val_loader is not None:
            model.eval()
            val_loss_total = 0.0
            with torch.no_grad():
                for batch_x, batch_y in val_loader:
                    batch_x = batch_x.to(device)
                    batch_y = batch_y.to(device)
                    preds = model(batch_x)
                    loss = nn.functional.mse_loss(preds, batch_y)
                    val_loss_total += loss.item() * batch_x.size(0)

            val_loss = val_loss_total / len(val_loader.dataset)
            history["val_loss"].append(val_loss)
            monitor_value = val_loss

            if val_loss < (best_val_loss - min_delta):
                best_val_loss = val_loss
                best_state = copy.deepcopy(model.state_dict())
                wait = 0
            else:
                if use_early_stopping:
                    wait += 1
                    if wait >= patience:
                        stop_now = True
        else:
            history["val_loss"].append(np.nan)
            monitor_value = train_loss
            best_val_loss = train_loss
            best_state = copy.deepcopy(model.state_dict())

        if scheduler is not None:
            scheduler.step(monitor_value)

        history["lr"].append(optimizer.param_groups[0]["lr"])

        if epoch % display_step == 0 or epoch == 1 or epoch == epochs:
            if val_loader is not None:
                print(f"{log_prefix}Epoch {epoch:3d} | Train Loss: {train_loss:.4f} | Val Loss: {monitor_value:.4f}")
            else:
                print(f"{log_prefix}Epoch {epoch:3d} | Train Loss: {train_loss:.4f}")

        if stop_now:
            print(f"{log_prefix}Early stopping at epoch {epoch:3d} (best val loss = {best_val_loss:.4f})")
            break

    if val_loader is not None:
        model.load_state_dict(best_state)
        return history, best_val_loss, best_state

    return history, history["train_loss"][-1], best_state


In [ ]:
# Hyperparameter Search
def normalize_hyperparameters(raw_params):
    normalized = {}
    for key, value in raw_params.items():
        if key in {"d_model", "nhead", "num_layers", "dim_feedforward", "batch_size"}:
            normalized[key] = int(value)
        elif key in {"dropout", "learning_rate"}:
            normalized[key] = float(value)
        else:
            normalized[key] = value
    return normalized

search_config = CONFIG.get("hyperparameter_search", {})
default_params = normalize_hyperparameters(CONFIG["default_hyperparameters"].copy())

if not isinstance(search_config, dict):
    raise ValueError("hyperparameter_search 必须是字典。")

param_space = search_config.get("param_space", {})
if param_space and not isinstance(param_space, dict):
    raise ValueError("hyperparameter_search['param_space'] 必须是字典。")

strategy = str(search_config.get("strategy", "bayesian")).lower()
if strategy not in {"bayesian", "grid"}:
    raise ValueError("hyperparameter_search['strategy'] 仅支持 'bayesian' 或 'grid'。")

if not rolling_folds:
    raise RuntimeError("至少需要一个滚动验证折才能进行超参数搜索。")

evaluation_cache = {}
trial_records = []


def evaluate_candidate(params, trial_idx=None, total_label=None):
    suffix = f"/{total_label}" if total_label else ""
    prefix = f"[Trial {trial_idx}{suffix}] " if trial_idx else ""
    params = normalize_hyperparameters(params)
    key = tuple(sorted(params.items()))

    if params["d_model"] % params["nhead"] != 0:
        print(f"{prefix}跳过无效组合（d_model 需被 nhead 整除）: {params}")
        evaluation_cache[key] = (copy.deepcopy(params), float("inf"), None)
        return evaluation_cache[key]

    if key in evaluation_cache:
        cached_params, cached_loss, cached_history = evaluation_cache[key]
        if math.isfinite(cached_loss):
            print(f"{prefix}重复评估组合: {cached_params} -> 平均验证损失 {cached_loss:.4f}")
        else:
            print(f"{prefix}重复评估无效组合: {cached_params}")
        return evaluation_cache[key]

    fold_losses = []
    fold_histories = []
    for fold_idx, fold in enumerate(rolling_folds, start=1):
        model = instantiate_model(params)
        optimizer = torch.optim.Adam(model.parameters(), lr=params["learning_rate"])
        scheduler = build_scheduler(optimizer)

        train_loader = DataLoader(fold["train_dataset"], batch_size=params["batch_size"], shuffle=True, drop_last=False)
        val_loader = DataLoader(fold["val_dataset"], batch_size=params["batch_size"], shuffle=False, drop_last=False)

        history, val_loss, _ = run_training(
            model,
            optimizer,
            train_loader,
            val_loader,
            CONFIG["epochs"],
            log_prefix=f"{prefix}[Fold {fold_idx}/{len(rolling_folds)}] ",
            scheduler=scheduler,
        )

        if val_loss is None or not math.isfinite(val_loss):
            val_loss = float("inf")

        fold_losses.append(val_loss)
        fold_histories.append(history)

    if not fold_losses:
        raise RuntimeError("滚动验证过程中未得到任何有效的验证结果。")

    mean_val_loss = float(np.mean(fold_losses))
    print(f"{prefix}平均验证损失: {mean_val_loss:.4f}")

    evaluation_cache[key] = (copy.deepcopy(params), mean_val_loss, fold_histories)
    return evaluation_cache[key]

if not param_space:
    print("未提供超参数搜索空间，使用默认超参数。")
    cached_params, val_loss, history = evaluate_candidate(default_params, trial_idx=1, total_label=1)
    trial_records.append((cached_params, val_loss, history))
elif strategy == "grid":
    candidate_overrides = list(ParameterGrid(param_space))
    total_trials = len(candidate_overrides)
    print(f"使用网格搜索，共 {total_trials} 组候选超参数。")

    for trial_idx, override in enumerate(candidate_overrides, start=1):
        params = default_params.copy()
        params.update(override)
        cached_params, val_loss, history = evaluate_candidate(params, trial_idx=trial_idx, total_label=total_trials)
        trial_records.append((cached_params, val_loss, history))
else:
    if gp_minimize is None or Categorical is None:
        raise ImportError("未安装 scikit-optimize，无法执行贝叶斯优化。请先运行 `pip install scikit-optimize`。")

    dimensions = []
    for name, values in param_space.items():
        if isinstance(values, list):
            if not values:
                raise ValueError(f"hyperparameter_search['param_space'][{name!r}] 的候选列表不能为空。")
            dimensions.append(Categorical(values, name=name))
        elif isinstance(values, tuple) and len(values) == 2:
            low, high = values
            if isinstance(low, int) and isinstance(high, int):
                if low > high:
                    raise ValueError(f"超参数 {name} 的整数区间下界需小于等于上界。")
                dimensions.append(Integer(low, high, name=name))
            else:
                if low >= high:
                    raise ValueError(f"超参数 {name} 的连续区间下界需小于上界。")
                prior = "log-uniform" if low > 0 else "uniform"
                dimensions.append(Real(low, high, prior=prior, name=name))
        else:
            raise ValueError(f"hyperparameter_search['param_space'][{name!r}] 需为列表或长度为 2 的元组。")

    n_calls = int(search_config.get("n_trials", 20))
    if n_calls <= 0:
        raise ValueError("hyperparameter_search['n_trials'] 必须为正整数。")

    n_initial_points = int(search_config.get("n_initial_points", min(5, n_calls)))
    n_initial_points = max(1, min(n_initial_points, n_calls))
    acq_func = str(search_config.get("acq_func", "EI"))

    print(f"使用贝叶斯优化，计划评估 {n_calls} 次候选组合。")

    @use_named_args(dimensions)
    def objective(**overrides):
        trial_idx = len(trial_records) + 1
        params = default_params.copy()
        params.update(overrides)
        cached_params, val_loss, history = evaluate_candidate(params, trial_idx=trial_idx, total_label=n_calls)
        trial_records.append((cached_params, val_loss, history))
        return val_loss

    gp_minimize(
        objective,
        dimensions=dimensions,
        n_calls=n_calls,
        n_initial_points=n_initial_points,
        acq_func=acq_func,
        random_state=CONFIG["seed"],
        verbose=False,
    )

    print(f"贝叶斯优化完成，共评估 {len(trial_records)} 次（唯一组合 {len(evaluation_cache)} 组）。")

if not trial_records:
    raise RuntimeError("所有超参数组合均无效，请检查 hyperparameter_search 配置。")

finite_trials = [record for record in trial_records if math.isfinite(record[1])]
if not finite_trials:
    raise RuntimeError("所有超参数组合均无效，请检查 hyperparameter_search 配置。")

best_params, best_val_loss, best_history = min(finite_trials, key=lambda item: item[1])
print(f"最佳超参数: {best_params} (平均验证 Loss = {best_val_loss:.4f})")


In [ ]:
# Final Training with Best Hyperparameters
train_val_df = pd.concat([train_df, val_df])
feature_scaler_final = StandardScaler().fit(train_val_df[feature_cols])
target_scaler_final = StandardScaler().fit(train_val_df[target_cols])

train_val_dataset = build_sequence_dataset(
    train_val_df,
    feature_scaler=feature_scaler_final,
    target_scaler=target_scaler_final,
    lookback=lookback,
    horizon=horizon,
    feature_cols=feature_cols,
    target_cols=target_cols,
    dataset_name="train_val",
)

test_context = train_val_df.tail(lookback)
test_dataset = build_sequence_dataset(
    test_df,
    feature_scaler=feature_scaler_final,
    target_scaler=target_scaler_final,
    lookback=lookback,
    horizon=horizon,
    feature_cols=feature_cols,
    target_cols=target_cols,
    context_df=test_context,
    dataset_name="test",
)

train_val_loader = DataLoader(train_val_dataset, batch_size=best_params["batch_size"], shuffle=True, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=best_params["batch_size"], shuffle=False, drop_last=False)

final_model = instantiate_model(best_params)
final_optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params["learning_rate"])
final_scheduler = build_scheduler(final_optimizer)

history, _, final_state = run_training(
    final_model,
    final_optimizer,
    train_val_loader,
    val_loader=None,
    epochs=CONFIG["epochs"],
    log_prefix="[Final] ",
    scheduler=final_scheduler,
)

final_model.load_state_dict(final_state)
target_scaler = target_scaler_final  # 用于后续反归一化

fold_train_counts = [len(fold["train_dataset"]) for fold in rolling_folds]
fold_val_counts = [len(fold["val_dataset"]) for fold in rolling_folds]
print(
    f"滚动验证折数: {len(rolling_folds)} | 平均训练序列: {np.mean(fold_train_counts):.1f} | 平均验证序列: {np.mean(fold_val_counts):.1f}"
)
print(f"最终训练序列数: {len(train_val_dataset)} | 测试序列数: {len(test_dataset)}")


In [ ]:
# Evaluation on Test Set
final_model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        preds = final_model(batch_x).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(batch_y.numpy())

if not all_preds:
    raise ValueError("测试集中没有可用于评估的序列。")

preds_scaled = np.concatenate(all_preds, axis=0)
targets_scaled = np.concatenate(all_targets, axis=0)

preds = target_scaler.inverse_transform(preds_scaled)
targets = target_scaler.inverse_transform(targets_scaled)

horizon = CONFIG["horizon"]
lookback = CONFIG["lookback"]

directional_threshold = float(CONFIG.get("directional_threshold", 0.0))

def compute_direction(delta, threshold):
    direction = np.sign(delta)
    if threshold > 0:
        flat_mask = np.abs(delta) <= threshold
        direction[flat_mask] = 0.0
    return direction

combined_targets_df = pd.concat([train_val_df[target_cols], test_df[target_cols]], axis=0)
combined_target_values = combined_targets_df.to_numpy()

num_samples = preds.shape[0]
start_position = lookback + horizon - 1
target_positions = np.arange(start_position, start_position + num_samples)
last_observation_positions = target_positions - horizon

if np.any(last_observation_positions <= 0):
    raise ValueError("随机游走评估需要至少两期的历史值，请增大窗口长度或检查数据切分。")

last_observations = combined_target_values[last_observation_positions]
previous_observations = combined_target_values[last_observation_positions - 1]
random_walk_preds = last_observations.copy()
baseline_direction_matrix = compute_direction(last_observations - previous_observations, directional_threshold)

print("随机游走基准使用上一期的实际值作为水平预测，并沿用上一期的涨跌方向作为方向预测。")

metrics = []
for idx, col in enumerate(target_cols):
    actual = targets[:, idx]
    predicted = preds[:, idx]
    last_obs = last_observations[:, idx]
    baseline_pred = random_walk_preds[:, idx]
    baseline_direction = baseline_direction_matrix[:, idx]

    actual_direction = compute_direction(actual - last_obs, directional_threshold)
    predicted_direction = compute_direction(predicted - last_obs, directional_threshold)

    direction_accuracy = (predicted_direction == actual_direction).mean()
    baseline_direction_accuracy = (baseline_direction == actual_direction).mean()

    metrics.append(
        {
            "Target": col,
            "Model": "Transformer",
            "MAE": mean_absolute_error(actual, predicted),
            "RMSE": root_mean_squared_error(actual, predicted),
            "R^2": r2_score(actual, predicted),
            "Directional Accuracy": direction_accuracy,
        }
    )
    metrics.append(
        {
            "Target": col,
            "Model": "Random Walk",
            "MAE": mean_absolute_error(actual, baseline_pred),
            "RMSE": root_mean_squared_error(actual, baseline_pred),
            "R^2": r2_score(actual, baseline_pred),
            "Directional Accuracy": baseline_direction_accuracy,
        }
    )

metrics_df = pd.DataFrame(metrics).set_index(["Target", "Model"])
display(metrics_df)

history_df = pd.DataFrame(history)
ax = history_df.dropna(axis=1, how="all").plot(title="Training Loss (Final Model)", figsize=(8, 4))
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
plt.show()


In [ ]:
# Prediction Plot
prediction_index = test_df.index[horizon - 1 : horizon - 1 + len(preds)]

result_columns = {}
for idx, col in enumerate(target_cols):
    result_columns[f"Actual_{col}"] = targets[:, idx]
    result_columns[f"Predicted_{col}"] = preds[:, idx]

result_df = pd.DataFrame(result_columns, index=prediction_index)

plot_cols = [f"Actual_{target_cols[0]}", f"Predicted_{target_cols[0]}"]
ax = result_df[plot_cols].plot(figsize=(12, 5), title=f"Actual vs. Predicted ({target_cols[0]})")
ax.set_xlabel("Date")
ax.set_ylabel(target_cols[0])
plt.show()

display(result_df.head())